# RCPGenerator Getting Started

RCPGenerator provides particle packing in Python for random close packing and hypersphere packing in flat and curved containers. This notebook translates the existing example scripts into notebook sections, including target packing fraction staging and inline visualization.

## Installation

This notebook clones the repository, moves into the Python package folder, and compiles/installs the C++ core and Python bindings.

In [ ]:
!git clone https://github.com/KD-physics/RCPGenerator.git
%cd RCPGenerator/python_code/python
!pip install .

Compilation may take a short time in Colab.

## Verify Install

In [ ]:
import rcpgenerator
print(rcpgenerator.Packing)

## Notebook Helpers

These helpers mirror the shared example utilities used by the existing scripts.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("examples/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PALETTE_MAP = {
    "2d_monodisperse_box": 1,
    "2d_bidisperse_box": 5,
    "2d_polydisperse_box": 10,
    "2d_circular_container": 12,
    "2d_powerlaw_phi_staging": 8,
    "3d_monodisperse_box": 2,
    "3d_bidisperse_box": 6,
    "3d_cylindrical_container": 9,
    "3d_spherical_container": 4,
    "starter": 7,
}

def distribution_summary(dist):
    dist_type = dist.get("type", "mono")
    if dist_type == "mono":
        return f"mono(d={dist.get('d')})"
    if dist_type == "bidisperse":
        return (
            f"bidisperse(d1={dist.get('d1')}, d2={dist.get('d2')}, "
            f"p={dist.get('p')})"
        )
    if dist_type == "lognormal":
        return f"lognormal(mu={dist.get('mu')}, sigma={dist.get('sigma')})"
    if dist_type == "flat":
        return f"flat(d_min={dist.get('d_min')}, d_max={dist.get('d_max')})"
    if dist_type == "custom":
        values = dist.get("custom", [])
        return f"custom(len={len(values)})"
    return dist_type

def walls_summary(walls):
    return "periodic" if all(int(w) == 0 for w in walls) else str(walls)

def print_case_summary(case_name, packing):
    print(f"case: {case_name}")
    print(f"N: {packing.N}")
    print(f"Ndim: {packing.Ndim}")
    print(f"box: {list(packing.box)}")
    print(f"walls: {walls_summary(list(packing.walls))}")
    print(f"distribution: {distribution_summary(dict(packing.dist))}")
    print(f"final phi: {packing.phi_final}")
    print(f"steps: {packing.steps}")
    print(f"force magnitude: {packing.force_magnitude}")
    print(f"max_min_dist: {packing.max_min_dist}")

def show_and_save(case_name, packing):
    palette_choice = PALETTE_MAP.get(case_name, 1)
    packing.show_packing(palette_choice=palette_choice)
    image_path = packing.savefig(
        path=OUTPUT_DIR / f"{case_name}.png",
        palette_choice=palette_choice,
    )
    print(f"image: {image_path}")
    return image_path


## `minimal.py`

Minimal 2D monodisperse box packing using the current class-based API.

In [ ]:
CASE_NAME = "2d_monodisperse_box"

packing = rcpgenerator.Packing(
    phi=0.11,
    N=250,
    Ndim=2,
    box=[1.0, 1.0],
    walls=[0, 0],
    fix_height=False,
    dist={"type": "mono", "d": 1.0},
    neighbor_max=0,
    seed=123,
)

packing.pack()
print_case_summary(CASE_NAME, packing)
show_and_save(CASE_NAME, packing)

## `starter.py`

Small starter case that shows the basic object lifecycle, pack call, and saved figure output.

In [ ]:
CASE_NAME = "starter"

packing = rcpgenerator.Packing(
    phi=0.11,
    N=4,
    Ndim=2,
    box=[1.0, 1.0],
    walls=[0, 0],
    fix_height=False,
    dist={"type": "mono", "d": 1.0},
    neighbor_max=0,
    seed=123,
)

print("needs_initialize before pack:", packing.needs_initialize)
print("needs_pack before pack:", packing.needs_pack)
result = packing.pack()
packing.show_packing(palette_choice=PALETTE_MAP[CASE_NAME])
image_path = packing.savefig(
    path=OUTPUT_DIR / "starter.png",
    palette_choice=PALETTE_MAP[CASE_NAME],
)
print("final phi:", result["phi"])
print("force samples:", len(result["force_history"]))
print("image:", image_path)

## `example_2d_bidisperse_box.py`

2D bidisperse packing in a periodic box.

In [ ]:
CASE_NAME = "2d_bidisperse_box"

packing = rcpgenerator.Packing(
    phi=0.11,
    N=250,
    Ndim=2,
    box=[1.0, 1.0],
    walls=[0, 0],
    fix_height=False,
    dist={"type": "bidisperse", "d1": 0.8, "d2": 1.2, "p": 0.5},
    neighbor_max=0,
    seed=124,
)

packing.pack()
print_case_summary(CASE_NAME, packing)
show_and_save(CASE_NAME, packing)

## `example_2d_circular_container.py`

2D monodisperse packing in the legacy circular-container wall convention.

In [ ]:
CASE_NAME = "2d_circular_container"

packing = rcpgenerator.Packing(
    phi=0.25,
    N=500,
    Ndim=2,
    box=[1.0, 1.0],
    walls=[-2, 0],
    fix_height=False,
    dist={"type": "mono", "d": 1.0},
    neighbor_max=0,
    seed=128,
)

packing.pack()
print_case_summary(CASE_NAME, packing)
show_and_save(CASE_NAME, packing)

## `example_2d_polydisperse_box.py`

2D lognormal polydisperse packing in a periodic box.

In [ ]:
CASE_NAME = "2d_polydisperse_box"

packing = rcpgenerator.Packing(
    phi=0.11,
    N=250,
    Ndim=2,
    box=[1.0, 1.0],
    walls=[0, 0],
    fix_height=False,
    dist={"type": "lognormal", "mu": 0.0, "sigma": 0.35},
    neighbor_max=0,
    seed=125,
)

packing.pack()
print_case_summary(CASE_NAME, packing)
show_and_save(CASE_NAME, packing)

## `example_2d_powerlaw_phi_staging.py`

2D power-law staging example that grows to a target packing fraction, then steps upward with fixed-diameter relaxation.

In [ ]:
CASE_NAME = "2d_powerlaw_phi_staging"

packing = rcpgenerator.Packing(
    phi=0.11,
    N=250,
    Ndim=2,
    box=[1.0, 1.0],
    walls=[0, 0],
    fix_height=False,
    dist={"type": "powerlaw", "d_min": 0.3, "d_max": 1.8, "exponent": -2.5},
    neighbor_max=0,
    seed=131,
)

packing.relax(n_steps=5000, target_phi=0.82)
for target_phi in [0.84, 0.86, 0.88, 0.90, 0.92, 0.94]:
    packing.update_phi(target_phi)
    packing.relax(n_steps=1000, fix_diameter=True)

print_case_summary(CASE_NAME, packing)
show_and_save(CASE_NAME, packing)

## `example_3d_bidisperse_box.py`

3D bidisperse packing in a periodic box.

In [ ]:
CASE_NAME = "3d_bidisperse_box"

packing = rcpgenerator.Packing(
    phi=0.08,
    N=250,
    Ndim=3,
    box=[1.0, 1.0, 1.0],
    walls=[0, 0, 0],
    fix_height=False,
    dist={"type": "bidisperse", "d1": 0.85, "d2": 1.15, "p": 0.5},
    neighbor_max=0,
    seed=127,
)

packing.pack()
print_case_summary(CASE_NAME, packing)
show_and_save(CASE_NAME, packing)

## `example_3d_cylindrical_container.py`

3D monodisperse packing in the legacy cylindrical-container wall convention.

In [ ]:
CASE_NAME = "3d_cylindrical_container"

packing = rcpgenerator.Packing(
    phi=0.25,
    N=500,
    Ndim=3,
    box=[1.0, 1.0, 1.0],
    walls=[-2, 0, 0],
    fix_height=False,
    dist={"type": "mono", "d": 1.0},
    neighbor_max=0,
    seed=129,
)

packing.pack()
print_case_summary(CASE_NAME, packing)
show_and_save(CASE_NAME, packing)

## `example_3d_monodisperse_box.py`

3D monodisperse packing in a periodic box.

In [ ]:
CASE_NAME = "3d_monodisperse_box"

packing = rcpgenerator.Packing(
    phi=0.08,
    N=250,
    Ndim=3,
    box=[1.0, 1.0, 1.0],
    walls=[0, 0, 0],
    fix_height=False,
    dist={"type": "mono", "d": 1.0},
    neighbor_max=0,
    seed=126,
)

packing.pack()
print_case_summary(CASE_NAME, packing)
show_and_save(CASE_NAME, packing)

## `example_3d_spherical_container.py`

3D monodisperse packing in the legacy spherical-container wall convention.

In [ ]:
CASE_NAME = "3d_spherical_container"

packing = rcpgenerator.Packing(
    phi=0.25,
    N=500,
    Ndim=3,
    box=[1.0, 1.0, 1.0],
    walls=[-3, 0, 0],
    fix_height=False,
    dist={"type": "mono", "d": 1.0},
    neighbor_max=0,
    seed=130,
)

packing.pack()
print_case_summary(CASE_NAME, packing)
show_and_save(CASE_NAME, packing)